In [11]:
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import sys
import json
import glob
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt

In [12]:
sys.path.insert(0, str(Path("../src").resolve()))
plt.ioff()
from model import GeosteeringHybridModel

In [13]:
BATCH_SIZE = 256

# Load the SAME normalization stats that were computed from the training
# data during training (see train_full.py / dataset.py),
norm_stats_path = Path("../src/models/normalization_stats.json")
if not norm_stats_path.exists():
    raise FileNotFoundError(
        f"Normalization stats file missing: {norm_stats_path.resolve()}. "
        "Run train_full.py first -- it saves this file next to the checkpoint."
    )
with open(norm_stats_path) as f:
    norm_stats = json.load(f)

WINDOW_SIZE = norm_stats["window_size"]
GR_MEAN = norm_stats["gr_mean"]
GR_STD = norm_stats["gr_std"]
Z_MEAN = norm_stats["z_mean"]
Z_STD = norm_stats["z_std"]
TVT_MEAN = norm_stats["tvt_mean"]
TVT_STD = norm_stats["tvt_std"]
USE_RELATIVE_Z = norm_stats.get("use_relative_z", True)
print(f"Loaded normalization stats: {norm_stats}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Inference running on: {device}")

# ---------------------------------------------------------------------
# 2. Load best model
# ---------------------------------------------------------------------
model_path = Path("../src/models/best_geosteering_model.pth")
if not model_path.exists():
    raise FileNotFoundError(f"Model file missing: {model_path.resolve()}")

model = GeosteeringHybridModel(num_features=2, window_size=WINDOW_SIZE)
model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
model.to(device)
model.eval()
print("Model successfully loaded.")

Loaded normalization stats: {'gr_mean': 88.37197262437826, 'gr_std': 23.47621122950059, 'z_mean': -9379.078995950811, 'z_std': 634.2401575966342, 'tvt_mean': 11492.89305007537, 'tvt_std': 637.5371038898802, 'window_size': 100, 'use_relative_z': False}
Inference running on: cuda
Model successfully loaded.


In [14]:
# ---------------------------------------------------------------------
# 3. Search for test files
# ---------------------------------------------------------------------
test_dir = Path("../data/test")
test_files = list(test_dir.glob("*__horizontal_well.csv"))
print(f"Found test wells: {len(test_files)}\n")

submission_data = []
start_time = time.perf_counter()

os.makedirs("img", exist_ok=True)

# ---------------------------------------------------------------------
# 4. Inference & Plot Loop
# ---------------------------------------------------------------------
with torch.inference_mode():
    for file_path in tqdm(test_files, desc="Process and plot wells", colour="green"):
        well_id = file_path.name.split("__")[0]
        df = pd.read_csv(file_path)

        # --- Data Preparation ---
        df["GR"] = pd.to_numeric(df["GR"], errors="coerce").ffill().bfill()
        df["Z"] = pd.to_numeric(df["Z"], errors="coerce").ffill().bfill()
        df["MD"] = pd.to_numeric(df["MD"], errors="coerce")  # Ensure MD for plot

        if USE_RELATIVE_Z:
            df['Z'] = df['Z'] - df['Z'].iloc[0]

        features = df[["GR", "Z"]].to_numpy(dtype=np.float32, copy=True)
        features[:, 0] = (features[:, 0] - GR_MEAN) / GR_STD
        features[:, 1] = (features[:, 1] - Z_MEAN) / Z_STD

        windows = np.lib.stride_tricks.sliding_window_view(features, window_shape=WINDOW_SIZE, axis=0)
        windows = np.ascontiguousarray(windows, dtype=np.float32)
        number_of_windows = len(windows)

        # --- Model Prediction ---
        window_preds = []
        for start_idx in range(0, number_of_windows, BATCH_SIZE):
            end_idx = min(start_idx + BATCH_SIZE, number_of_windows)
            x_batch = torch.from_numpy(windows[start_idx:end_idx]).to(device)
            preds = model(x_batch).reshape(-1).cpu().numpy()
            window_preds.append(preds)

        window_preds = np.concatenate(window_preds).astype(np.float64)

        # --- De-scaling ---
        window_preds = (window_preds * TVT_STD) + TVT_MEAN

        predictions = np.full(len(df), np.nan, dtype=np.float64)
        predictions[WINDOW_SIZE - 1:] = window_preds
        df["TVT_pred"] = predictions

        # ANCHORING
        missing_mask = df["TVT_input"].isna()
        known_mask = ~missing_mask

        if known_mask.any() and missing_mask.any():
            last_known_idx = df[known_mask].index[-1]
            last_known_tvt = df.loc[last_known_idx, "TVT_input"]
            model_val_at_anchor = df.loc[last_known_idx, "TVT_pred"]

            if pd.notna(model_val_at_anchor):
                shift = last_known_tvt - model_val_at_anchor
                # Apply the shift only to the missing (blind) predictions
                df.loc[missing_mask, "TVT_pred"] += shift

        # --- Plotting for each well (normally with Matplotlib) ---
        fig, ax = plt.subplots(figsize=(12, 6))

        # Green line: Known TVT
        ax.plot(df['MD'], df['TVT_input'], color='green', linewidth=2, label='Known TVT_input')
        # Purple line: Model prediction
        ax.plot(df['MD'], df['TVT_pred'], color='purple', linestyle='--', linewidth=2, label='Model Prediction (TVT)')

        # Determine red area for blind flight
        missing_mask = df["TVT_input"].isna()
        if missing_mask.any():
            eval_start_idx = missing_mask.idxmax()
            eval_start_md = float(df.loc[eval_start_idx, 'MD'])
            max_md = float(df['MD'].max())

            ax.axvline(x=eval_start_md, color='red', linestyle='-', alpha=0.5, label='Start blind flight (NaN)')
            ax.axvspan(eval_start_md, max_md, color='red', alpha=0.1)

        ax.set_title(f'Geosteering Prediction for Well {well_id}')
        ax.set_xlabel('Measured Depth (MD)')
        ax.set_ylabel('True Vertical Thickness (TVT)')
        ax.legend()
        ax.grid(True, alpha=0.3)
        fig.tight_layout()

        # Save and cleanly remove this VERY SPECIFIC image from memory
        fig.savefig(f'img/inference_plot_{well_id}_epoch.png', dpi=100, bbox_inches='tight')
        plt.close(fig)

        # --- Collect Kaggle Submission Data ---
        for idx in df[missing_mask].index:
            submission_id = f"{well_id}_{idx}"
            tvt_value = df.loc[idx, "TVT_pred"]

            if pd.isna(tvt_value):
                tvt_value = TVT_MEAN

            submission_data.append({
                "id": submission_id,
                "tvt": tvt_value
            })

Found test wells: 3



Process and plot wells: 100%|██████████| 3/3 [00:01<00:00,  1.85it/s]


In [15]:
# ---------------------------------------------------------------------
# 6. Create and save Submission CSV
# ---------------------------------------------------------------------
submission_df = pd.DataFrame(submission_data)

# index=False is extremely important for Kaggle, otherwise an extra column is generated
submission_df.to_csv("submission.csv", index=False)

total_time = time.perf_counter() - start_time
print(f"\nDone! The file 'submission.csv' was successfully created in {total_time:.1f} seconds.")
print(f"Number of missing values predicted for Kaggle: {len(submission_df)}")


Done! The file 'submission.csv' was successfully created in 1.7 seconds.
Number of missing values predicted for Kaggle: 14151
